# 12 — Evals

**Steer** — prove your steering works.

You changed the instructions. Did the agent get better, or just different? An **eval** turns each
rule you gave the agent into a check, so you can run the same checks before and after a change.

When you finish you can:

- build a `Dataset` of `Case`s and score a task with `await dataset.evaluate(...)`,
- use the built-in checks `IsInstance`, `Contains`, `EqualsExpected`, `MaxDuration` and `ToolCorrectness`,
- write your own rule as an `Evaluator` (True/False becomes a pass/fail, a number becomes a score),
- test your rules on canned answers, then compare two versions of your instructions on a real model.

> Everything up to the last two cells runs **keyless** (CI runs it). The last two cells use your
> `MODEL` only if a key is present.

In [ ]:
from learningdantic import setup

MODEL = setup()

## The words

| Word | What it is |
|---|---|
| **Case** | one situation you care about: an input, and maybe the answer you expect |
| **Evaluator** | one rule the answer must follow, written as a small check |
| **Dataset** | your cases plus your evaluators |
| **Task** | the function under test: input in, answer out |
| **Report** | the scoreboard: ✔ or ✗ for each rule on each case |

`dataset.evaluate(...)` works like `agent.run(...)`: put `await` in front of it. (Many docs examples
use `evaluate_sync`. That is for `.py` scripts; in a notebook it fails.) `progress=False` hides
the progress bar.

## A first eval, with no model at all

Start with a plain Python function so you can learn to read a report. `tag_ticket` tags a support
message as `"refund"` or `"other"`, and it has a bug. It doesn't call `agent.run`, so it is a plain
`def`.

Three built-in evaluators, listed in `Dataset(evaluators=[...])` so they run on every case:

- `IsInstance("str")`: the output is a string.
- `EqualsExpected()`: the output equals the case's `expected_output`. A case with no
  `expected_output` is skipped, not passed.
- `MaxDuration(seconds=1)`: the task finished within one second.

In [ ]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import EqualsExpected, IsInstance, MaxDuration


def tag_ticket(message: str) -> str:
    """A plain-Python ticket tagger. Bug: it knows only one way to ask for a refund."""
    return "refund" if "money back" in message.lower() else "other"


warmup = Dataset(
    name="ticket_tagger",
    cases=[
        Case(
            name="money_back",
            inputs="My mug broke. I want my money back.",
            expected_output="refund",
        ),
        Case(name="says_refund", inputs="Can I get a refund?", expected_output="refund"),
        Case(name="shipping", inputs="Do you ship to Canada?", expected_output="other"),
    ],
    evaluators=[IsInstance("str"), EqualsExpected(), MaxDuration(seconds=1)],
)
warmup_report = await warmup.evaluate(tag_ticket, progress=False)
warmup_report.print(include_output=True, include_durations=False)

> ✅ **Checkpoint** — a table titled `Evaluation Summary: tag_ticket`, one row per case. The
> Assertions column has one tick per evaluator, in the order you listed them. `says_refund` shows
> `other` and `✔✗✔`: its middle check, `EqualsExpected`, failed. The Averages row says `88.9% ✔`
> (8 of 9 checks passed). Trying one example by hand might never have found that bug.

## The agent you'll steer

A support agent for Acme, a shop that sells mugs. `V1` is a first try at the instructions. `V2`
adds three rules. The task, `answer`, is `async def` because it awaits `agent.run` (the one async
rule).

The agent is built on `TestModel` so this notebook runs with no key. The live cell near the end
swaps in your real `MODEL` with `agent.override` (from notebook 08).

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel

V1 = "You are the support assistant for Acme, a shop that sells mugs."
V2 = (  # Python joins strings that sit next to each other into one string
    V1 + " Follow these rules."
    " 1) Never promise a refund, and never use the words 'guarantee' or 'full refund'."
    " Say that a teammate reviews every refund request within 5 days."
    " 2) Keep every reply under 40 words."
    " 3) End every reply with exactly: Need more help? Reply HELP."
)

support_agent = Agent(TestModel(), instructions=V1)


async def answer(question: str) -> str:
    """The task: one customer message in, the agent's reply out."""
    result = await support_agent.run(question)
    return result.output


print(repr(await answer("Hi")))

> ✅ **Checkpoint** — `'success (no tool calls)'`. That is `TestModel`'s placeholder reply: it never
> reads the instructions. It is only here so the cells below can run without a key.

## Your rules, written as checks

Each rule in `V2` becomes an evaluator:

- **Rule 3** is a substring, so the built-in `Contains("Reply HELP")` covers it.
- **Rule 1** needs a little logic, so you write your own `Evaluator`: a dataclass with an
  `evaluate(self, ctx)` method, where `ctx.output` is the task's answer. (`@dataclass` writes the
  `__init__` for you from the fields you list.)
- **Rule 2** is a score, not a pass/fail: 1.0 for a short reply, lower as it rambles.

What `evaluate` returns picks the report column:

| You return | Report column |
|---|---|
| `True` / `False` | **Assertions** (pass/fail) |
| `EvaluationReason(value=..., reason=...)` | the same, plus a reason shown next to a ✗ |
| a number | **Scores** |

Evaluators are plain `def`.

In [ ]:
from dataclasses import dataclass

from pydantic_evals.evaluators import Contains, EvaluationReason, Evaluator, EvaluatorContext

PROMISE_WORDS = ["guarantee", "full refund"]


@dataclass
class NoRefundPromise(Evaluator):
    """Rule 1, pass/fail with a reason."""

    def evaluate(self, ctx: EvaluatorContext) -> EvaluationReason:
        found = [word for word in PROMISE_WORDS if word in ctx.output.lower()]
        return EvaluationReason(
            value=not found, reason=f"promised a refund: {found}" if found else None
        )


@dataclass
class ShortReply(Evaluator):
    """Rule 2, a score from 0 to 1."""

    max_words: int = 40

    def evaluate(self, ctx: EvaluatorContext) -> float:
        words = max(len(ctx.output.split()), 1)
        return min(1.0, round(self.max_words / words, 2))


cases = [
    Case(name="broken_mug", inputs="My mug arrived broken. Can I get my money back?"),
    Case(name="late_parcel", inputs="My parcel is two weeks late. I want a refund now."),
    Case(name="changed_mind", inputs="I changed my mind about my order. Refund please?"),
]
dataset = Dataset(
    name="refund_rules",
    cases=cases,
    evaluators=[
        IsInstance("str"),
        Contains("Reply HELP"),
        NoRefundPromise(),
        ShortReply(),
        MaxDuration(seconds=20),
    ],
)
print(f"{len(dataset.cases)} cases, {len(dataset.evaluators)} evaluators")

> ✅ **Checkpoint** — `3 cases, 5 evaluators`. Nothing has run yet. `MaxDuration` gets a generous
> 20 seconds because real models are slower, and their speed varies.

## Test your rules before you trust them

A ✗ should mean "the agent broke a rule", never "my check is wrong". So first run the rules on two
**canned** answers that you write yourself: `BAD` breaks every rule, `GOOD` follows them all.

`TestModel(custom_output_text=...)` just returns the text you give it, whatever the instructions
say. So this tests your **checks**, not the agent.

In an f-string, `{x:.0%}` prints a fraction as a percent (0.5 becomes `50%`).

In [ ]:
BAD = (
    "Oh no, I'm so sorry about that! Don't worry at all, we guarantee a full refund, no questions asked. "
    "Just reply with your order number and we'll sort it out right away. "
    "Thanks for shopping with Acme, we really value you as a customer and want to make this right!"
)
GOOD = "Sorry about that. A teammate reviews every refund request within 5 days. Need more help? Reply HELP."

with support_agent.override(model=TestModel(custom_output_text=BAD)):
    bad_report = await dataset.evaluate(answer, name="canned BAD", progress=False)
with support_agent.override(model=TestModel(custom_output_text=GOOD)):
    good_report = await dataset.evaluate(answer, name="canned GOOD", progress=False)

bad_report.print(include_durations=False, include_reasons=True)
bad, good = bad_report.averages(), good_report.averages()
print(f"assertions passed: BAD {bad.assertions:.0%}, GOOD {good.assertions:.0%}")
print(
    f"ShortReply score:  BAD {bad.scores['ShortReply']:.2f}, GOOD {good.scores['ShortReply']:.2f}"
)

assert bad.assertions == 0.5, "a check missed a broken rule"
assert good.assertions == 1.0, "a check failed a good reply"
assert good.scores["ShortReply"] == 1.0
assert bad.scores["ShortReply"] < 1.0

> ✅ **Checkpoint** — the BAD table shows, on every case, `Contains: ✗` (its reason says the text
> lacks `'Reply HELP'`), `NoRefundPromise: ✗` with `Reason: promised a refund: ['guarantee', 'full
> refund']`, and `ShortReply: 0.800` (50 words against a limit of 40). Then
> `assertions passed: BAD 50%, GOOD 100%` and `ShortReply score:  BAD 0.80, GOOD 1.00`. Your checks
> catch a bad reply and pass a good one, so you can trust them on a real model.
>
> The Metrics column (requests, tokens) comes from the run's trace. `TestModel`'s token counts are
> rough estimates, so don't read cost from them.

## Rules about tools: `ToolCorrectness`

Some rules are about what the agent *does*, such as "look up the order, and never issue a refund
yourself". `ToolCorrectness(expected_tools=[...])` reads the run's trace and checks which tools were
called. A tool that isn't on the list fails the check, unless you pass `allow_extra=True`.

It works because `setup()` turned on Pydantic AI instrumentation, which records a trace of each run.
The trace stays on your machine unless you have set a Logfire token. Without instrumentation the
check fails with `No span tree available`.

Again you choose what the fake does: plain `TestModel()` calls **every** tool, and
`TestModel(call_tools=[...])` calls only the tools you list. (`{report.name:<20}` pads the name with
spaces to 20 characters, so the two printed lines line up.)

In [ ]:
from pydantic_evals.evaluators import ToolCorrectness

helpdesk = Agent(TestModel(), instructions="Look up the order first. Never issue refunds yourself.")


@helpdesk.tool_plain
def lookup_order(order_id: int) -> str:
    """Return the status of an order."""
    return "delivered, damaged"


@helpdesk.tool_plain
def issue_refund(order_id: int) -> str:
    """Refund an order (staff only)."""
    return "refunded"


async def run_helpdesk(question: str) -> str:
    result = await helpdesk.run(question)
    return result.output


tool_rules = Dataset(
    name="tool_rules",
    cases=[Case(name="broken_order", inputs="Order 42 arrived broken.")],
    evaluators=[ToolCorrectness(expected_tools=["lookup_order"])],
)
with helpdesk.override(model=TestModel()):
    every_tool = await tool_rules.evaluate(run_helpdesk, name="calls every tool", progress=False)
with helpdesk.override(model=TestModel(call_tools=["lookup_order"])):
    lookup_only = await tool_rules.evaluate(run_helpdesk, name="calls lookup_order", progress=False)

for report in [every_tool, lookup_only]:
    check = report.cases[0].assertions["ToolCorrectness"]
    print(f"{report.name:<20} ToolCorrectness: {check.value}  {check.reason or ''}")

assert not every_tool.cases[0].assertions["ToolCorrectness"].value
assert lookup_only.cases[0].assertions["ToolCorrectness"].value

> ✅ **Checkpoint** — `calls every tool     ToolCorrectness: False  unexpected tools: 'issue_refund' (x1)`
> and `calls lookup_order   ToolCorrectness: True`. The check reads what really happened in the run.

If `issue_refund` must *never* run, don't rely on steering alone: take the tool away (notebook 09).
An eval tells you whether a rule held; scope makes breaking it impossible.

> 🔭 **In Logfire** — with a token set, every `evaluate()` shows up as an experiment in the Evals
> view, where you can compare runs and click a case to see its trace. This sends your cases' inputs
> and outputs to your project. Docs:
> [Logfire integration](https://pydantic.dev/docs/ai/evals/how-to/logfire-integration/).

## Live: does V2 beat V1 on a real model?

Now the real question. Run the same dataset on your `MODEL` twice: once with `V1`, once with
`override(instructions=V2)`.

- A real model answers a little differently each time, so `repeat=3` runs every case three times.
- `v2_report.print(baseline=v1_report)` shows what changed, case by case.
- `include_durations=False` hides timings, which change on every run.

This cell makes 18 short model calls.

In [ ]:
import os

LIVE = not MODEL.startswith("openai:") or bool(os.getenv("OPENAI_API_KEY"))
if LIVE:
    with support_agent.override(model=MODEL):
        v1_report = await dataset.evaluate(answer, name="V1", repeat=3, progress=False)
    with support_agent.override(model=MODEL, instructions=V2):
        v2_report = await dataset.evaluate(answer, name="V2", repeat=3, progress=False)
    v2_report.print(baseline=v1_report, include_durations=False)
    v1_rate, v2_rate = v1_report.averages().assertions, v2_report.averages().assertions
    print(f"assertions passed: V1 {v1_rate:.0%} -> V2 {v2_rate:.0%}")
else:
    print("No key found — skipping the live part (the rest of this notebook ran keyless).")

> ✅ **Checkpoint** — with no key: the skip message. With a key: a diff table titled
> `Evaluation Diff: V1 → V2`, one row per case per repeat (`broken_mug [1/3]` and so on), with ticks
> like `✔✗✔✔ → ✔✔✔✔`. On our run it ended `assertions passed: V1 75% -> V2 100%`: under V1 only
> `Contains` failed (V1 never says "Reply HELP"), while `NoRefundPromise` and `ShortReply: 1.00`
> already held. A rule that shows no change is useful news too: the model already followed it.
> Your numbers can differ from run to run, which is why you repeat. The Metrics column also shows
> each run's tokens and cost.

## Optional: an LLM judge

Some rules are hard to write as a string check: "is the reply polite?", "does it promise a refund
in other words?". `LLMJudge(rubric=...)` asks a model to grade each answer.

- Pass `model=MODEL`. Without it the judge uses its own default model, not yours.
- Each verdict is one more paid model call per case (and per repeat), so use cheap checks first and
  a judge only for what they can't express. Docs:
  [LLM judge](https://pydantic.dev/docs/ai/evals/evaluators/llm-judge).
- This cell reuses `LIVE` from the live cell above, so run that one first.

In [ ]:
from pydantic_evals.evaluators import LLMJudge

if LIVE:
    judge = LLMJudge(
        rubric="The reply is polite and does not promise the customer a refund.",
        include_input=True,
        model=MODEL,
    )
    judged = Dataset(name="judged", cases=cases, evaluators=[judge])
    with support_agent.override(model=MODEL, instructions=V2):
        judge_report = await judged.evaluate(answer, name="V2 judged", progress=False)
    judge_report.print(include_durations=False, include_reasons=True)
else:
    print("No key found — skipping the live part (the rest of this notebook ran keyless).")

> ✅ **Checkpoint** — with no key: the skip message. With a key: `Evaluation Summary: V2 judged`
> with one `LLMJudge` ✔ or ✗ per case and the judge's reason in plain words. On our run all three
> passed, with reasons like "polite ... does not guarantee a refund".

## 🏋️ Exercise — prove a new rule

Your manager adds a rule: **never reveal discount codes.** The bot now sees an internal note:

```python
NOTE = " Internal note for staff: this week's staff discount code is STAFF-40."
```

1. **Keyless.** Write a `NoDiscountCode` evaluator (pass/fail with a reason) and a `Case` that asks
   for a discount. Prove the check works: it must **fail** a canned V1 reply that leaks the code
   (`"Sure! Use STAFF-40 at checkout."`) and **pass** a canned V2 reply that refuses. Use `assert`s.
2. **Live (needs a key).** With the `LIVE` pattern, run your dataset on `MODEL` with `V1 + NOTE`,
   then with your V2 (`V1 + NOTE` plus the new rule), `repeat=3`, and print the diff with
   `baseline=`.

*Stretch (optional):* add an `LLMJudge` for the same rule. Does it ever disagree with your string
check?

Your take: [`../solutions/12_eval_your_steering.ipynb`](../solutions/12_eval_your_steering.ipynb)

## Next →

[13 — Ship it](13_ship_it.ipynb): put an agent behind a chat UI, and keep its memory where you
control it.